# Gold Price Prediction

**Tabular Regression Project** — Predict gold prices from financial indicators.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Gold price data (local CSV)
Target: Gold price

## 2 · Project Overview

We predict **gold prices** using a local dataset of financial indicators and gold price history. Gold is a major commodity and safe-haven asset — predicting its price has applications in portfolio management, risk hedging, and macroeconomic analysis.

## 3 · Learning Objectives

1. Work with commodity/financial datasets.
2. Handle correlated financial features.
3. Apply regression models to price prediction.
4. Use LazyPredict and FLAML for benchmarking.
5. Understand the limits of financial prediction.

## 4 · Problem Statement

Predict the **gold price** from related financial indicators such as silver price, stock indices, oil prices, and currency exchange rates.

## 5 · Why This Project Matters

- **Gold price prediction** is relevant for investors, central banks, and commodity traders.
- Teaches working with highly correlated financial features.
- Demonstrates the interplay between commodities, stocks, and currencies.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Local file: `gold_price.csv` |
| **Target** | GLD (Gold ETF price) or similar gold price column |
| **Features** | Financial indicators (SPX, USO, SLV, EUR/USD, etc.) |

## 7 · Dataset Source and License Notes

- **Source**: Gold price dataset, commonly from Kaggle.
- **License**: Educational use.
- **Note**: Financial data — not for investment decisions without proper validation.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Loading

In [ ]:
data_file = os.path.join(SAVE_DIR, 'gold_price.csv')
assert os.path.exists(data_file), f'Not found: {data_file}'
df = pd.read_csv(data_file)
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

## 13 · Exploratory Data Analysis

In [ ]:
# Identify target
target_candidates = [c for c in df.columns if any(kw in c.upper() for kw in ['GLD', 'GOLD'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    TARGET = num_cols[-1]

print(f'Target: {TARGET}')

# Drop date column
for col in df.columns:
    if 'date' in col.lower():
        df = df.drop(columns=[col])

num_cols = df.select_dtypes(include='number').columns.tolist()
n_plot = min(4, len(num_cols))
fig, axes = plt.subplots(1, n_plot, figsize=(4*n_plot, 4))
if n_plot == 1: axes = [axes]
for ax, col in zip(axes, num_cols[:n_plot]):
    df[col].hist(bins=30, ax=ax, edgecolor='black')
    ax.set_title(col[:15])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

# Correlation with target
corr = df.select_dtypes(include='number').corr()[TARGET].drop(TARGET).sort_values(ascending=False)
print(f'\nCorrelation with {TARGET}:')
print(corr)

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
# Drop non-numeric columns
for col in df.select_dtypes(include='object').columns:
    df = df.drop(columns=[col])

# Fill missing
for col in df.columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.4f}, MAE={baseline_mae:.4f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Silver (SLV)** is typically the strongest predictor — precious metals move together.
- **Oil prices (USO)** and **S&P 500 (SPX)** provide complementary signals.
- **EUR/USD** captures currency effects on dollar-denominated gold.
- These relationships reflect how gold acts as a safe-haven asset.

## 26 · Limitations

- Financial correlations are not stable over time.
- No macroeconomic indicators (inflation, interest rates, GDP).
- Random train/test split — time-based split would be more realistic.
- Historical patterns may not persist in future market regimes.

## 27 · How to Improve

1. Use walk-forward validation.
2. Add macroeconomic indicators (CPI, Fed Funds Rate).
3. Include volatility indices (VIX).
4. Add lagged features for temporal dynamics.
5. Try target differencing for stationarity.

## 28 · Production Considerations

- Financial prediction models require extensive backtesting.
- Regulatory disclosure if used for investment advice.
- Model drift monitoring crucial in financial markets.
- Ensemble with fundamental analysis.

## 29 · Common Mistakes

1. Using random splits for time-series financial data.
2. Confusing correlation with causation.
3. Not accounting for look-ahead bias.
4. Over-interpreting high R² on correlated financial data.

## 30 · Mini Challenge

1. Add lag features (previous day's price).
2. Use walk-forward cross-validation.
3. Predict daily returns instead of price levels.
4. Add a VIX feature and evaluate impact.

## 31 · Final Summary

- Gold price is highly predictable from related financial instruments (silver, oil, stocks).
- High R² reflects genuine cross-market correlations.
- Production use requires time-based validation and regime-change monitoring.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')